In [3]:
from setup import setup_tools, data_path, aig_path, verilog_path, cnf_path, aag_path
from tools_fns import verilog_to_aig, aig_miter, aig_to_dimacs, aig_cec, cadical_check, aig_to_aag

import os

setup_tools()

Setting project root: /home/krishnendu/fv-invariant-mining
Updated PATH to include: [PosixPath('/home/krishnendu/fv-invariant-mining/.tools/cadical/build'), PosixPath('/home/krishnendu/fv-invariant-mining/.tools/abc'), PosixPath('/home/krishnendu/fv-invariant-mining/.tools/oss-cad-suite/bin')]


In [4]:
for width in range(1, 15):
	print(f"Processing {width}-bit multiplier...")

	# Update the Verilog file with the current bit width
	with open(verilog_path / "multipliers.v", "r") as f:
		verilog_content = f.read()
	
	verilog_content = verilog_content.replace("{{BIT_WIDTH}}", str(width))
	
	# Write the updated content to a temporary file (to avoid modifying the original template)
	temp_verilog_file = verilog_path / f"multipliers_{width}bit.v"
	with open(temp_verilog_file, "w") as f:
		f.write(verilog_content)
	
	# Generate AIG files for both the unrolled and behavioral multiplier modules
	top_module_unrolled = f"unrolled_mult_{width}bit"
	top_module_behavioral = f"behavioral_mult_{width}bit"
	verilog_to_aig(temp_verilog_file, "unrolled_mult", aig_path/f"{top_module_unrolled}.aig")
	verilog_to_aig(temp_verilog_file, "behavioral_mult", aig_path/f"{top_module_behavioral}.aig")

	# Remove the temporary Verilog file
	os.remove(temp_verilog_file)

	aig_to_aag(aig_path/f"{top_module_unrolled}.aig", aag_path/f"{top_module_unrolled}.aag")
	aig_to_aag(aig_path/f"{top_module_behavioral}.aig", aag_path/f"{top_module_behavioral}.aag")

	aig_miter(aig_path/f"unrolled_mult_{width}bit.aig", aig_path/f"behavioral_mult_{width}bit.aig", aig_path/f"miter_mult_{width}bit.aig")
	aig_to_dimacs(aig_path/f"miter_mult_{width}bit.aig", cnf_path/f"miter_mult_{width}bit.cnf")
	aig_to_aag(aig_path/f"miter_mult_{width}bit.aig", aag_path/f"miter_mult_{width}bit.aag")

	aig_to_dimacs(aig_path/f"unrolled_mult_{width}bit.aig", cnf_path/f"unrolled_mult_{width}bit.cnf")
	aig_to_dimacs(aig_path/f"behavioral_mult_{width}bit.aig", cnf_path/f"behavioral_mult_{width}bit.cnf")

Processing 1-bit multiplier...
Processing 2-bit multiplier...


Processing 3-bit multiplier...
Processing 4-bit multiplier...
Processing 5-bit multiplier...
Processing 6-bit multiplier...
Processing 7-bit multiplier...
Processing 8-bit multiplier...
Processing 9-bit multiplier...
Processing 10-bit multiplier...
Processing 11-bit multiplier...
Processing 12-bit multiplier...
Processing 13-bit multiplier...
Processing 14-bit multiplier...


In [3]:
# Use ABC to perform CEC
width = 7
out = aig_cec(aig_path/f"unrolled_mult_{width}bit.aig", aig_path/f"behavioral_mult_{width}bit.aig")
print(out.output)

======== ABC command line "
	read /home/krishnendu/fv-invariant-mining/data/circuits/aig/unrolled_mult_7bit.aig;
	cec /home/krishnendu/fv-invariant-mining/data/circuits/aig/behavioral_mult_7bit.aig
	"
Networks are equivalent.  Time =     2.58 sec



In [5]:
# Use CaDiCaL to check satisfiability of the miter CNF
width = 8

sat, out = cadical_check(cnf_path/f"miter_mult_{width}bit.cnf")
print(f"CaDiCaL result for miter_mult_{width}bit.cnf: {sat}")
print(out.output)

CaDiCaL result for miter_mult_8bit.cnf: UNSATISFIABLE
c --- [ banner ] -------------------------------------------------------------
c 
c CaDiCaL SAT Solver
c Copyright (c) 2016-2025
c A. Biere, M. Fleury, N. Froleyks, K. Fazekas, F. Pollitt, T. Faller
c JKU Linz, University of Freiburg, TU Wien
c 
c Version 3.0.0 7b99c07f0bcab5824a5a3ce62c7066554017f641
c g++ (GCC) 15.2.1 20260123 (Red Hat 15.2.1-7) -Wall -Wextra -O3 -DNDEBUG
c Sat Feb 21 12:26:06 IST 2026 Linux kfed 6.18.8-100.fc42.x86_64 x86_64
c 
c --- [ parsing input ] ------------------------------------------------------
c 
c reading DIMACS file from '/home/krishnendu/fv-invariant-mining/data/circuits/cnf/miter_mult_8bit.cnf'
c opening file to read '/home/krishnendu/fv-invariant-mining/data/circuits/cnf/miter_mult_8bit.cnf'
c found 'p cnf 679 2553' header
c parsed 2553 clauses in 0.00 seconds process time
c 
c --- [ options ] ------------------------------------------------------------
c 
c all options are set to their default v